# Building a From-Scratch RAG Notebook for PDF URL Lists

## Introduction

This notebook builds an end-to-end Retrieval-Augmented Generation (RAG) pipeline for PDF documents. Give it direct PDF URLs either in the `pdf_urls` Python list or in a UTF-8 text file with one URL per line. The notebook downloads each PDF, extracts text page by page, chunks the text without crossing page boundaries, embeds the chunks, and stores them in Chroma for retrieval and answer generation.

The URLs must point to the PDFs themselves, not to HTML pages that list or link to PDFs. Blank lines and lines beginning with `#` are ignored in the URL-list file. The downloader checks the host's `robots.txt` before fetching each PDF.

## 1) PDF URL input and ingestion

Add direct PDF links to `pdf_urls` in the ingestion cell, or create `pdf_urls.txt` in the notebook's working directory and put one direct PDF URL on each line. If both are provided, both sources are ingested. For example:

```text
https://example.org/guidance.pdf
https://example.org/forms/application.pdf
```

The notebook uses `pypdf` to extract selectable text. Scanned/image-only PDFs need OCR before they can be indexed. Extracted chunks retain their source URL and page number as Chroma metadata.

## 2) Required Python libraries

Install these packages in the notebook's active Python environment if needed:

```python
%pip install openai python-dotenv requests beautifulsoup4 chromadb pypdf
```

`openai` sends embedding and chat requests through OpenRouter. Set `OPENROUTER_API_KEY` in the project's `.env` file before running embedding or answer-generation cells. Chroma persists the local index in `./slc_chroma_db`.

## 3) Embeddings and vector storage

Each page-aware text chunk is embedded with `openai/text-embedding-3-small` and upserted into the `slc_documents` Chroma collection. Stable chunk IDs make rerunning ingestion update existing chunks rather than add duplicates.

## 4) Retrieval, RRF, and answer generation

The notebook retrieves semantically similar chunks from Chroma, demonstrates Reciprocal Rank Fusion (RRF), and sends the top three chunks with the question to an OpenRouter chat model. The answer prompt instructs the model to say when the supplied context does not contain the answer.


In [5]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise ValueError("Set OPENROUTER_API_KEY in the project .env file.")

# This function takes a text chunk and returns its vector embedding.
def get_embedding(text_chunk: str, model: str = "openai/text-embedding-3-small") -> list[float]:
    """
    Generates a vector embedding for a given text chunk using an OpenAI-compatible API.
    """
    # Clean up the text by replacing newlines, which can affect embedding quality.
    text_chunk = text_chunk.replace("\n", " ")
    
    # Initialize the client to connect to the custom base URL.
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=api_key,
    )
    
    # Call the embeddings endpoint.
    response = client.embeddings.create(
        model=model,
        input=[text_chunk] # The input needs to be a list of strings.
    )
    
    # Extract the embedding vector from the response.
    return response.data[0].embedding

In [6]:
import hashlib
import time
from io import BytesIO
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser

import requests
from pypdf import PdfReader


def load_pdf_urls(urls: list[str] | None = None, url_list_path: str | None = None) -> list[str]:
    """Combine direct PDF URLs and one-URL-per-line text-file input."""
    loaded_urls = list(urls or [])

    if url_list_path and os.path.isfile(url_list_path):
        with open(url_list_path, encoding="utf-8") as url_file:
            loaded_urls.extend(
                line.strip()
                for line in url_file
                if line.strip() and not line.lstrip().startswith("#")
            )

    return list(dict.fromkeys(url.strip() for url in loaded_urls if url.strip()))


def chunk_page_text(text: str, max_words: int = 350, overlap_words: int = 50) -> list[str]:
    """Split one PDF page into overlapping word-based chunks."""
    if max_words <= 0 or overlap_words < 0 or overlap_words >= max_words:
        raise ValueError("Require max_words > overlap_words >= 0")

    words = text.split()
    step = max_words - overlap_words
    return [
        " ".join(words[start : start + max_words])
        for start in range(0, len(words), step)
    ]


def fetch_pdf_pages(url: str, robots_cache: dict[str, RobotFileParser]) -> list[tuple[int, str]]:
    """Download a permitted PDF URL and return its extractable text by page."""
    parsed_url = urlparse(url)
    if parsed_url.scheme not in {"http", "https"} or not parsed_url.netloc:
        raise ValueError(f"Provide a complete http(s) PDF URL: {url}")

    user_agent = "RAGLearningNotebook/1.0"
    headers = {"User-Agent": user_agent}
    origin = f"{parsed_url.scheme}://{parsed_url.netloc}"

    if origin not in robots_cache:
        robots_url = f"{origin}/robots.txt"
        robots_response = requests.get(robots_url, headers=headers, timeout=10)
        if robots_response.status_code in {401, 403, 429} or robots_response.status_code >= 500:
            raise RuntimeError(f"Could not safely check robots.txt for {url}")

        robots = RobotFileParser()
        robots.set_url(robots_url)
        robots.parse(
            robots_response.text.splitlines() if robots_response.status_code < 400 else []
        )
        robots_cache[origin] = robots

    if not robots_cache[origin].can_fetch(user_agent, url):
        raise PermissionError(f"robots.txt disallows fetching this URL: {url}")

    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    if not response.content.startswith(b"%PDF-"):
        content_type = response.headers.get("Content-Type", "").lower()
        raise ValueError(f"URL did not return PDF content: {url} ({content_type})")

    reader = PdfReader(BytesIO(response.content))
    if reader.is_encrypted:
        raise ValueError(f"Encrypted PDFs are not supported: {url}")

    pages = [
        (page_number, page.extract_text() or "")
        for page_number, page in enumerate(reader.pages, start=1)
    ]
    if not any(text.strip() for _, text in pages):
        raise ValueError(f"No selectable text found in PDF; scanned PDFs need OCR: {url}")
    return pages


def ingest_pdf_urls(pdf_urls: list[str], batch_size: int = 64) -> int:
    """Fetch, page-chunk, embed, and upsert direct PDF URLs into Chroma."""
    if not pdf_urls:
        raise ValueError("Add at least one direct PDF URL before running ingestion.")
    if batch_size <= 0:
        raise ValueError("batch_size must be greater than zero")

    embedding_client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=api_key,
    )
    robots_cache: dict[str, RobotFileParser] = {}
    stored_count = 0

    for pdf_index, url in enumerate(pdf_urls):
        pages = fetch_pdf_pages(url, robots_cache)
        source_hash = hashlib.sha256(url.encode("utf-8")).hexdigest()[:16]
        page_chunks = [
            (page_number, chunk)
            for page_number, page_text in pages
            for chunk in chunk_page_text(page_text)
        ]

        for start in range(0, len(page_chunks), batch_size):
            batch = page_chunks[start : start + batch_size]
            documents = [chunk for _, chunk in batch]
            response = embedding_client.embeddings.create(
                model="openai/text-embedding-3-small",
                input=documents,
            )
            embeddings = [
                item.embedding for item in sorted(response.data, key=lambda item: item.index)
            ]
            chunk_indexes = range(start, start + len(batch))
            collection.upsert(
                ids=[
                    f"{source_hash}-p{page_number}-c{chunk_index}"
                    for chunk_index, (page_number, _) in zip(chunk_indexes, batch)
                ],
                documents=documents,
                embeddings=embeddings,
                metadatas=[
                    {"source": url, "page_number": page_number, "chunk_index": chunk_index}
                    for chunk_index, (page_number, _) in zip(chunk_indexes, batch)
                ],
            )
            stored_count += len(batch)

        if pdf_index < len(pdf_urls) - 1:
            time.sleep(1)

    return stored_count

### Creating a Local Vector Store with Chroma

A vector store, or vector database, indexes our embeddings to enable rapid similarity searches. Chroma is an excellent choice for local development because it's easy to set up and can persist data to disk.

The process involves creating a "collection" (similar to a table in a SQL database), and then adding our chunks, their embeddings, and their metadata to it [citation: 71].

In [7]:
import chromadb

print("ChromaDB imported successfully.")
client = chromadb.PersistentClient(path="./slc_chroma_db")
collection_name = "slc_pdf_documents"
collection = client.get_or_create_collection(name=collection_name)

ChromaDB imported successfully.


In [8]:
# Option 1: add direct PDF URLs here (not HTML pages linking to PDFs).
pdf_urls = [
     "https://practitioners.slc.co.uk/media/2188/residency-lle-v10.pdf",
     "https://practitioners.slc.co.uk/media/2194/lle-guidance-chapter-tuition-fee-loan-entitlement-v10.pdf",
     "https://practitioners.slc.co.uk/media/2186/dsa-guidance-2627-v20.pdf",
     "https://practitioners.slc.co.uk/media/2197/sfe-assessing-eligibility-guidance-ay-2627-v20.pdf",
]

# Option 2: add one direct PDF URL per line in this text file.
pdf_url_list_path = "pdf_urls.txt"
pdf_urls = load_pdf_urls(pdf_urls, pdf_url_list_path)

if pdf_urls:
    stored_count = ingest_pdf_urls(pdf_urls)
    print(
        f"Upserted {stored_count} PDF chunks from {len(pdf_urls)} URLs; "
        f"the PDF collection now has {collection.count()} chunks."
    )
else:
    print(
        "Add direct PDF links to pdf_urls above or create pdf_urls.txt "
        "with one PDF URL per line, then rerun this cell."
    )

Upserted 842 PDF chunks from 4 URLs; the PDF collection now has 842 chunks.


### Retrieving Top-K Chunk Candidates

Once the vector store is populated, we can query it to find the chunks most semantically similar to a user's question. This is the "Retrieval" part of RAG. We will retrieve a set of `k` candidates, which will be passed to the re-ranking step later.

In [9]:
def retrieve_top_k_chunks(query: str, k: int = 10) -> list:
    """
    Retrieves the top-k most relevant chunks from the Chroma vector store.
    We retrieve a larger k=10 here to provide a good candidate pool for RRF re-ranking.
    """
    if collection.count() == 0:
        raise ValueError(
            "The Chroma collection is empty. Add document chunks and embeddings before retrieval."
        )

    # First, generate an embedding for the user's query.
    query_embedding = get_embedding(query)
    
    # Query the collection to find the k most similar documents.
    # The 'results' object contains the IDs, documents, distances, and metadatas.
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )
    
    return results

# Example usage:
user_query = "What is LLE?"
# Retrieve 10 candidates for re-ranking
candidate_chunks = retrieve_top_k_chunks(user_query, k=10)

# The output `candidate_chunks` is a dictionary-like object that can be
# processed in the next step (Reciprocal Rank Fusion).
print(f"Retrieved {len(candidate_chunks['ids'][0])} candidates for the query.")

Retrieved 10 candidates for the query.


This completes the embedding and initial retrieval phase. We now have a ranked list of candidate chunks based on vector similarity, ready for the more advanced re-ranking and generation stages.

---
## 4) RRF (Reciprocal Rank Fusion) + LLM integration using your OpenRouter OpenAI-compatible code

While vector similarity search is powerful, its results can be significantly improved by incorporating other ranking signals. This final section demonstrates how to implement **Reciprocal Rank Fusion (RRF)** to create a more robust ranking of retrieved documents and then use the top results to generate a final, context-aware answer with the specified Large Language Model (LLM).

### Required Python Libraries for this Section

This stage integrates the retrieved chunks with the LLM. We will primarily use the `openai` library as specified by the user to connect to the OpenRouter endpoint.

| Library | Purpose | Installation Command |
| :--- | :--- | :--- |
| `openai` | To interact with the OpenAI-compatible API for the final answer generation step. | `pip install openai` |
| `os` | A built-in library used to securely access environment variables like API keys. | (Built-in) |

### Enhancing Relevance with Reciprocal Rank Fusion (RRF)

Reciprocal Rank Fusion (RRF) is a simple yet highly effective technique for combining multiple ranked lists of results into a single, fused list. It is particularly useful in RAG for hybrid search, where you might have one list from a dense vector search (like we've done) and another from a sparse keyword search (like BM25).

The RRF formula calculates a new score for each document based on its rank in each list:
`RRF_Score(doc) = Σ (1 / (k + rank))`

-   **`rank`** is the position of the document in a given results list (e.g., 1st, 2nd, 3rd).
-   **`k`** is a constant, typically set to 60, which helps to de-emphasize the impact of documents with very low ranks.

For this learning notebook, we will simulate having two result lists to demonstrate the fusion mechanism. In a production system, these could come from different retrieval methods [citation: 83].

In [11]:
def reciprocal_rank_fusion(ranked_lists: list[list], k: int = 60) -> dict:
    """
    Performs Reciprocal Rank Fusion on a list of ranked document lists.
    
    Args:
        ranked_lists: A list where each element is a list of document IDs in ranked order.
        k: A constant for the RRF formula. Defaults to 60.
        
    Returns:
        A dictionary of document IDs and their fused RRF scores.
    """
    fused_scores = {}
    
    # Iterate through each ranked list.
    for ranked_list in ranked_lists:
        # Iterate through each document and its rank in the current list.
        # The rank is its index + 1.
        for rank, doc_id in enumerate(ranked_list, 1):
            if doc_id not in fused_scores:
                fused_scores[doc_id] = 0
            # Add the reciprocal rank score to the document's fused score.
            fused_scores[doc_id] += 1 / (k + rank)
            
    return fused_scores

### End-to-End Workflow: From Query to Final Answer

Now, we'll tie everything together. This complete workflow reads a query from a file, retrieves initial candidates, re-ranks them using RRF, selects the top 3, and passes them to the LLM to generate an answer.

In [14]:
# Import the user-provided function and other necessary components.
from openai import OpenAI
import os

# --- Assume previous functions are defined in the notebook ---
# get_embedding(text_chunk)
# retrieve_top_k_chunks(query, k)
# reciprocal_rank_fusion(ranked_lists, k)
# -------------------------------------------------------------

# User-provided code for the LLM API connection.
def get_ai_response(prompt: str) -> str:
    """
    Connects to the OpenRouter API to get a response from the specified LLM.
    """
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_API_KEY"),
    )
    completion = client.chat.completions.create(
        model="openai/gpt-3.5-turbo", # Using a more common model for demonstration
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    return completion.choices[0].message.content

# --- Main Workflow ---

# 1. Get the user query from a file.
query_file_path = "user_query.txt"
with open(query_file_path, "w") as f:
    f.write("Can you please list all LLE key definitions?")

with open(query_file_path, "r") as f:
    user_query = f.read().strip()
print(f"Read query from file: '{user_query}'\n")

# 2. Retrieve initial candidates from the vector store.
# We get k=10 candidates to provide a good pool for RRF.
retrieved_results = retrieve_top_k_chunks(user_query, k=10)
dense_ranked_ids = retrieved_results['ids'][0]

# --- RRF Simulation ---
# In a real system, you'd have another list from a different retriever (e.g., BM25).
# For this notebook, we'll simulate a second list by slightly reordering the first one.
sparse_ranked_ids = dense_ranked_ids[2:6] + dense_ranked_ids[0:2] + dense_ranked_ids[6:]
print(f"Initial dense ranking (top 5): {dense_ranked_ids[:5]}")
print(f"Simulated sparse ranking (top 5): {sparse_ranked_ids[:5]}\n")

# 3. Apply RRF to get fused scores.
fused_scores = reciprocal_rank_fusion([dense_ranked_ids, sparse_ranked_ids])

# 4. Sort the documents by their new fused scores to get the final ranking.
final_ranked_ids = sorted(fused_scores.keys(), key=lambda x: fused_scores[x], reverse=True)
print(f"Final RRF-ranked IDs (top 5): {final_ranked_ids[:5]}\n")

# 5. Get the top 3 chunks based on the final RRF ranking.
top_3_ids = final_ranked_ids[:3]
# Retrieve the full document content for the top 3 chunks.
top_chunks_data = collection.get(ids=top_3_ids, include=["documents"])
top_chunks_text = top_chunks_data['documents']

# 6. Construct the prompt for the LLM.
# This prompt template clearly separates the context from the question.
context_str = "\n\n---\n\n".join(top_chunks_text)
prompt = f"""
Based on the following context from the Student Loans Company UK, please answer the user's question.
If the answer is not contained within the context provided, state that you cannot answer based on the given information.

CONTEXT:
{context_str}

QUESTION:
{user_query}
"""
print("--- PROMPT FOR LLM ---")
print(prompt)
print("----------------------\n")

# 7. Call the LLM to generate the final answer.
final_answer = get_ai_response(prompt)

print("--- FINAL ANSWER ---")
print(final_answer)
print("--------------------")

Read query from file: 'Can you please list all LLE key definitions?'

Initial dense ranking (top 5): ['36ffce9767975485-p7-c8', 'cd843bb76d87246c-p3-c2', '36ffce9767975485-p2-c1', 'cd843bb76d87246c-p2-c1', 'cd843bb76d87246c-p6-c8']
Simulated sparse ranking (top 5): ['36ffce9767975485-p2-c1', 'cd843bb76d87246c-p2-c1', 'cd843bb76d87246c-p6-c8', 'cd843bb76d87246c-p22-c39', '36ffce9767975485-p7-c8']

Final RRF-ranked IDs (top 5): ['36ffce9767975485-p2-c1', '36ffce9767975485-p7-c8', 'cd843bb76d87246c-p2-c1', 'cd843bb76d87246c-p3-c2', 'cd843bb76d87246c-p6-c8']

--- PROMPT FOR LLM ---

Based on the following context from the Student Loans Company UK, please answer the user's question.
If the answer is not contained within the context provided, state that you cannot answer based on the given information.

CONTEXT:
2 OFFICIAL OFFICIAL Purpose This document sets out guidance on the eligible regulatory categories and the residency requirements that students must meet to be eligible for funding un

This notebook implements a PDF-focused RAG workflow: load direct PDF URLs from a Python list or a text file, extract selectable text page by page, create overlapping chunks within each page, embed and store them in a dedicated Chroma collection, retrieve relevant passages, and generate a context-grounded answer. Chunks retain their source URL and page number so retrieved information can be traced back to the PDF. Scanned PDFs require OCR before they can be indexed.